In [1]:
%%capture
!pip install kaggle

In [ ]:
import os
import shutil
import cv2
import json
import subprocess
import logging
import math
from pathlib import Path
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
from PIL import Image

class FrameExtractor:
    def __init__(self, input_path: str, output_path: str, kaggle_api: dict, base_dataset_name: str, progress_info: dict = {}):
        """
        Initializes the FrameExtractor.

        Args:
            input_path (str): Path to the directory containing input videos.
            output_path (str): Path to the directory to save extracted frames.
            kaggle_api (dict): Dictionary containing Kaggle API 'username' and 'key'.
            base_dataset_name (str): Base name for the datasets to be created on Kaggle.
            progress_info (dict, optional): Dictionary to control partial processing. 
                                            Defaults to {}. If empty, all videos will be processed.
                                            Example: {"start_index": 0, "end_index": 100}
        """
        self.input_path = Path(input_path)
        self.output_path = Path(output_path)
        self.upload_dir = self.output_path / "upload"
        self.progress_info = progress_info or {}

        self.kaggle_username = kaggle_api.get("username")
        self.kaggle_key = kaggle_api.get("key")
        if not self.kaggle_username or not self.kaggle_key:
            raise ValueError("Kaggle API 'username' and 'key' must not be empty.")

        # Setup Kaggle API credentials
        kaggle_json_path = Path.home() / ".kaggle" / "kaggle.json"
        kaggle_json_path.parent.mkdir(parents=True, exist_ok=True)
        with open(kaggle_json_path, "w") as f:
            json.dump({"username": self.kaggle_username, "key": self.kaggle_key}, f)
        os.chmod(kaggle_json_path, 0o600)

        self.base_dataset_name = base_dataset_name
        self.target_height = 0

        # Clean up and create necessary directories
        print(f"Cleaning up output directory '{self.output_path}'...")
        shutil.rmtree(self.output_path, ignore_errors=True)
        self.output_path.mkdir(parents=True, exist_ok=True)
        self.upload_dir.mkdir(exist_ok=True)

    def get_video_files(self, file_extension: str = '.mp4') -> list:
        """
        Gets a list of video files from the input path.
        Filters the list based on self.progress_info if provided.
        """
        file_list = sorted([Path(root) / f for root, _, files in os.walk(self.input_path) for f in files if f.endswith(file_extension)])
        print(f"Found a total of {len(file_list)} video files with extension '{file_extension}'.")

        if not self.progress_info:
            print("No progress info provided, processing all files.")
            self.progress_info = {
                "start_index": 0,
                "end_index": len(file_list)
            }

        start_index = self.progress_info.get("start_index", 0)
        end_index = self.progress_info.get("end_index", len(file_list))
        
        print(f"Processing files from index {start_index} to {end_index}.")
        return file_list[start_index:end_index]

    def _get_video_metadata(self, video_path: Path) -> dict:
        """
        Lấy metadata bằng ffprobe và CHỈ trả về:
          {"fps": <float|None>, "duration": <float|None>}
        """
        import json, subprocess
        from fractions import Fraction
    
        cmd = [
            'ffprobe', '-v', 'error',
            '-select_streams', 'v:0',
            '-show_entries', 'stream=avg_frame_rate,r_frame_rate:format=duration',
            '-of', 'json', str(video_path)
        ]
        out = subprocess.run(cmd, capture_output=True, text=True, check=True).stdout
        info = json.loads(out)
    
        s = (info.get('streams') or [{}])[0]
        f = info.get('format') or {}
    
        def to_fps(x):
            try:
                if not x or x == '0/0': return 0.0
                fr = Fraction(x)
                return float(fr) if fr.denominator != 0 else 0.0
            except Exception:
                return 0.0
    
        fps = to_fps(s.get('avg_frame_rate')) or to_fps(s.get('r_frame_rate')) or None
    
        duration = None
        try:
            d = f.get('duration')
            duration = float(d) if d not in (None, 'N/A') else None
        except Exception:
            duration = None
    
        return {"fps": fps, "duration": duration}
    
    # print(_get_video_metadata(Path("/kaggle/input/lucifer-v8/Videos_K07/video/K07_V030.mp4")))
    def _process_single_video(self, video_path: Path) -> Path:
        """
        Trích xuất frame bằng FFmpeg sang JPG (q=2, 4:4:4 nếu hỗ trợ),
        LẤY MỖI `frame_step` KHUNG, scale theo chiều cao mục tiêu (không padding),
        rồi đổi tên file theo đúng chỉ số frame gốc (0, step, 2*step, ...).
    
        Ghi metadata.json chỉ gồm {"fps", "duration"}.
        """
        import json, subprocess
        from pathlib import Path
    
        # Thư mục output cho video này
        video_name_stem = video_path.stem.replace(' ', '_')
        frame_out_dir = self.output_path / video_name_stem
        frame_out_dir.mkdir(parents=True, exist_ok=True)
    
        # Metadata (chỉ fps, duration)
        metadata = self._get_video_metadata(video_path)
        with open(frame_out_dir / 'metadata.json', 'w') as f:
            json.dump(metadata, f, indent=4)
    
        # frame_step (mặc định 7)
        step = getattr(self, 'frame_step', 7)
        try:
            step = max(1, int(step))
        except Exception:
            step = 7
    
        # target height (ưu tiên self.target_height, fallback self.height)
        target_h = getattr(self, 'target_height', None)
        try:
            target_h = int(target_h) if target_h is not None else None
        except Exception:
            target_h = None
    
        # ----- Filter graph -----
        # chọn các khung n % step == 0
        vf_parts = [f"select=not(mod(n\\,{step}))"]
        # scale theo chiều cao, giữ tỉ lệ (không padding)
        if target_h is not None and target_h > 0:
            vf_parts.append(f"scale=-1:{target_h}:flags=lanczos")
        vf_arg = ['-vf', ','.join(vf_parts)]
    
        # Helper chạy ffmpeg với 1 pix_fmt
        def _run_ffmpeg(pix_fmt: str):
            cmd = [
                'ffmpeg',
                '-v', 'error',
                '-nostdin',
                '-y',
                '-hwaccel', 'none',      # ép software decode (AV1/H.264 đều ổn)
                '-threads', '0',
                '-i', str(video_path),
                '-map', '0:v:0',
                '-vsync', '0',           # không thêm/bớt frame
                '-start_number', '0',    # xuất 0.jpg, 1.jpg, 2.jpg, ...
                *vf_arg,
                '-c:v', 'mjpeg',
                '-q:v', '1',             # chất lượng cao (1 tốt nhất ↔ 31 xấu nhất)
                '-pix_fmt', pix_fmt,     # ưu tiên 4:4:4, fallback 4:2:0
                str(frame_out_dir / '%d.jpg'),
            ]
            subprocess.run(cmd, check=True)
    
        # Thử 4:4:4, nếu lỗi thì fallback 4:2:0
        try:
            _run_ffmpeg('yuvj444p')
        except subprocess.CalledProcessError:
            _run_ffmpeg('yuvj420p')
    
        # ----- Đổi tên file theo đúng chỉ số frame gốc -----
        # 0.jpg -> 0.jpg ; 1.jpg -> step.jpg ; 2.jpg -> 2*step.jpg ; ...
        # Chạy ngược từ n về 0 để tránh xung đột tên file
        jpgs = sorted(frame_out_dir.glob('*.jpg'), key=lambda p: int(p.stem))
        for i in reversed(range(len(jpgs))):
            p = jpgs[i]
            target_idx = i * step
            p.rename(frame_out_dir / f"{target_idx}.jpg")
    
        return frame_out_dir

    def _get_dir_size_in_gb(self, dir_path: Path) -> float:
        """Calculates the total size of a directory in GB."""
        total_size = sum(f.stat().st_size for f in dir_path.glob('**/*') if f.is_file())
        return total_size / (1024 ** 3)

    def _create_new_dataset(self, dataset_name: str, message: str):
        """Creates a new dataset on Kaggle."""
        dataset_slug = dataset_name.lower().replace(' ', '-')
        dataset_id = f"{self.kaggle_username}/{dataset_slug}"
        print(f"Creating new dataset: {dataset_id}")
        dataset_meta = {
            "title": dataset_name,
            "id": dataset_id,
            "licenses": [{"name": "CC0-1.0"}]
        }
        with open(self.upload_dir / 'dataset-metadata.json', 'w') as f:
            json.dump(dataset_meta, f, indent=4)

        command = ["kaggle", "datasets", "create", "-p", str(self.upload_dir), "--dir-mode", "zip"]
        try:
            print(f"Executing command: {' '.join(command)}")
            subprocess.run(command, check=True, capture_output=True, text=True, encoding='utf-8')
            print(f"Successfully created dataset {dataset_id}!")
        except subprocess.CalledProcessError as e:
            print("\nError creating dataset:")
            print("Stdout:", e.stdout)
            print("Stderr:", e.stderr)
            raise e

    def extract_multi_video(self, batch_size: int, target_height: int = 480):
        """
        Extracts frames from multiple videos in batches and uploads them to Kaggle.
        
        Args:
            batch_size (int): Number of videos to process per batch.
            target_height (int): Target height for resizing frames while maintaining aspect ratio.
        """
        self.target_height = target_height
        list_video = self.get_video_files()
        if not list_video:
            print("No videos found to process.")
            return

        num_batches = math.ceil(len(list_video) / batch_size)
        print(f"Total videos will be split into {num_batches} batches, with up to {batch_size} videos per batch.")
        print(f"Frames will be resized to height: {target_height}px (width will vary based on aspect ratio)")

        for i in range(num_batches):
            batch_suffix = f"-batch-{i+1:03d}"
            dataset_name = self.base_dataset_name + batch_suffix
            print(f"\n{'='*20} Starting Batch {i + 1}/{num_batches} ({dataset_name}) {'='*20}")

            start_index = i * batch_size
            end_index = start_index + batch_size
            current_batch_videos = list_video[start_index:end_index]
            
            # Delete the upload directory before each batch to ensure it's empty
            shutil.rmtree(self.upload_dir, ignore_errors=True)
            self.upload_dir.mkdir(exist_ok=True)

            with Pool(cpu_count()) as p:
                iterator = p.imap_unordered(self._process_single_video, current_batch_videos)
                for processed_folder_path in tqdm(iterator, total=len(current_batch_videos), desc=f"Processing Batch {i+1}/{num_batches}"):
                    # Archive the frame directory into a zip file in the upload directory
                    zip_filename_base = self.upload_dir / processed_folder_path.name
                    shutil.make_archive(str(zip_filename_base), 'zip', str(processed_folder_path))
                    # Delete the frame directory after archiving
                    shutil.rmtree(processed_folder_path)

            batch_size_gb = self._get_dir_size_in_gb(self.upload_dir)
            print(f"Finished processing Batch {i + 1}. Total zip files size: {batch_size_gb:.4f} GB.")
            if batch_size_gb == 0:
                print("Warning: Batch is empty, no data to upload. Skipping this batch.")
                continue

            upload_message = f"Creating dataset {dataset_name} from Batch {i+1}/{num_batches}."
            self._create_new_dataset(dataset_name, upload_message)

        print(f"\n{'='*20} ALL BATCHES COMPLETED {'='*20}")

In [ ]:
import json, os
from pathlib import Path

INPUT_VIDEOS_PATH = '/kaggle/input/lucifer-v6/Videos_K18'
OUTPUT_PATH = '/kaggle/temp'
KAGGLE_API_CONFIG = {"username":"trandiep2105","key":"e4b793444a83cefa3b1aa1a0957b8f30"} # diep.003
DATASET_NAME = "lucifer-fn-300-6"
BATCH_SIZE=1000
TARGET_HEIGHT = 300  # Only height parameter, width will be calculated based on aspect ratio

# option A: run with start and end index
# progress_config = {
#     "start_index": 0,
#     "end_index": 1000,
# }

# option B: run all
progress_config = {}

try:
    # Khởi tạo lớp FrameExtractor với cấu hình tiến trình đã chọn
    extractor = FrameExtractor(
        input_path=INPUT_VIDEOS_PATH,
        output_path=OUTPUT_PATH,
        kaggle_api=KAGGLE_API_CONFIG,
        base_dataset_name=DATASET_NAME,
        progress_info=progress_config  # Truyền cấu hình vào đây
    )
    
    # Bắt đầu quá trình trích xuất với chỉ tham số chiều cao
    extractor.extract_multi_video(
        batch_size=BATCH_SIZE,
        target_height=TARGET_HEIGHT  # Chỉ cần tham số chiều cao
    )
    
except Exception as e:
    print(f"Một lỗi không mong muốn đã xảy ra: {e}")

Cleaning up output directory '/kaggle/temp'...
Found a total of 28 video files with extension '.mp4'.
No progress info provided, processing all files.
Processing files from index 0 to 28.
Total videos will be split into 1 batches, with up to 1000 videos per batch.
Frames will be resized to height: 300px (width will vary based on aspect ratio)

==================== Starting Batch 1/1 (lucifer-fn-300-6-batch-001) ====================


Processing Batch 1/1:  89%|████████▉ | 25/28 [08:50<00:23,  7.99s/it]  